### Example Fixed Project

The rest of this tutorial uses pre compiled ORBIT configs that are stored as .yaml files in the '~/configs/ folder. There are load and save methods available in ORBIT for working with .yaml files. These example projects each exhibit different functionalities within ORBIT. Using these examples and combinations of them, most project configurations can be modeled. 

In [1]:
import os
import pandas as pd
from ORBIT import ProjectManager, load_config

weather = pd.read_csv("data/era5_40.0N_72.5W_1990_2020.csv", parse_dates=["datetime"])\
            .set_index("datetime")

ParserError: while parsing a block mapping
  in "C:\ORBIT_procurement_by_year\ORBIT-2020\ORBIT\ORBIT\core\defaults\common_costs.yaml", line 2, column 1
expected <block end>, but found '<block mapping start>'
  in "C:\ORBIT_procurement_by_year\ORBIT-2020\ORBIT\ORBIT\core\defaults\common_costs.yaml", line 64, column 2

### Load the project configuration

In [ ]:
fixed_config = load_config("configs/example_fixed_project_600MW.yaml")  # Configs can be loaded with absolute or relative paths

print(type(fixed_config))                                         # They are loaded in as dictionaries.

print(f"Num turbines: {fixed_config['plant']['num_turbines']}")   # Once a configuration is loaded, different parameters can  
print(f"Turbine: {fixed_config['turbine']}")                      # be accessed using dict access.
print(f"\nSite: {fixed_config['site']}")

### Phases

This fixed project represents a generic Offshore Wind farm with 50 6MW turbines. It includes 5 design modules and 6 installation modules as seen below. This is a common set of modules to run for a fixed bottom project. This config will model the procurement and installation of monopiles, scour protection, array system, export system, offshore substation and the turbines.

In [ ]:
print(f"Design phases: {fixed_config['design_phases']}")
print(f"\nInstall phases: {list(fixed_config['install_phases'].keys())}")

### Run

This project is always being modeled with the example weather project supplied that is representative of US East Coast wind farm locations.

In [ ]:
project = ProjectManager(fixed_config, weather=weather)
project.run()

### Top Level Outputs

ProjectManager offers several high level result categories:
- Installation CapEx
- System CapEx (procurement of BOS subcomponents)
- Turbine CapEx
- Soft CapEx (project management costs)
- Total CapEx
- Total installation time
- etc.

In [ ]:
print(f"Installation CapEx:  {project.installation_capex/1e6:.0f} M")
print(f"System CapEx:        {project.system_capex/1e6:.0f} M")
print(f"Turbine CapEx:       {project.turbine_capex/1e6:.0f} M")
print(f"Soft CapEx:          {project.soft_capex/1e6:.0f} M")
print(f"Total CapEx:        {project.total_capex/1e6:.0f} M")

print(f"\nInstallation Time: {project.installation_time:.0f} h")

### CapEx Breakdown

In [ ]:
# The breakdown of project costs by module is available  at 'capex_breakdown'
data = project.capex_detailed_soft_capex_breakdown_per_kw

# Convert the dictionary into a pandas DataFrame
df = pd.DataFrame(list(data.items()), columns=['Category', 'Value'])

# Add a "Total" row
total_row = pd.DataFrame([['Total', df['Value'].sum()]], columns=['Category', 'Value'])
df = pd.concat([df, total_row], ignore_index=True)

# Display the DataFrame
df.to_csv("fixed_costs.csv")
display(df)

### Installation Actions

In [ ]:
df = pd.DataFrame(project.actions)    # The project simulation logs are also available for all modules
df

In [ ]:
# These logs can be sorted by phase by using DataFrame operations

turbine_install = df.loc[df['phase']=="TurbineInstallation"]
turbine_install

In [ ]:
# Operations can also be grouped to see a total amount of time spend on each operation

turbine_install.groupby(["action"]).sum()['duration']